# GFS 2 m temperature — NWP quickstart

Fetch a recent GFS run's 2 m temperature over a small bbox and read the resulting Cloud-Optimized GeoTIFF.

**Requirements** (live download):

```bash
pip install earthlens[nwp]
conda install -c conda-forge eccodes libgdal-grib   # binary libs
```

`herbie-data` downloads only the `.idx` byte range for the requested variable (>99 % bandwidth saving); `pyramids.grib.open_grib` reads it with GDAL's GRIB driver, and earthlens crops it to the bbox.

## Setup

A couple of imports: `datetime` to pick a recent cycle date, and the unified `EarthLens` entry point that drives the NWP backend.

In [ ]:
import datetime as dt

from earthlens.core import EarthLens

## Pick a recent cycle

GFS keeps roughly the last 10 days of runs on NODD, so we target yesterday's `00Z` cycle to be safe.

In [ ]:
# A recent 00Z cycle (GFS keeps ~10 days on NODD).
cycle_day = (dt.datetime.now(dt.UTC) - dt.timedelta(days=1)).strftime('%Y-%m-%d')
cycle_day

## Build the download request

Describe the request to `EarthLens`: the `nwp` backend, the `gfs` dataset, the 2 m temperature variable, the date window, a small bbox over the US Northeast, the output directory, lead step `0`, and the `aws` mirror.

In [ ]:
lens = EarthLens(
    data_source='nwp',
    dataset='gfs',
    variables=['temperature_2m'],
    start=cycle_day,
    end=cycle_day,
    aoi=[-80, 40, -75, 45],
    path='out/gfs',
    steps=[0],
    mirror='aws',
)

## Download the COGs

`download()` fetches each cycle/step as a Cloud-Optimized GeoTIFF cropped to the bbox and returns the written paths. We list their file names to confirm what landed on disk.

In [ ]:
paths = lens.download(progress_bar=False)
[p.name for p in paths]

## Read one output

The output is a standard COG, so we open the first one with `pyramids.grib.open_grib` and inspect its grid shape (bands, rows, columns).

In [ ]:
from pyramids.grib import open_grib

ds = open_grib(str(paths[0]))
ds.shape

## Next steps

To request several lead times, pass `steps=[0, 6, 12, 24]` (or `horizon=48`); each `(cycle, step)` yields one COG. Add `aggregate=AggregationConfig(freq="1D", op="mean")` to reduce the stack to daily means.